# DSMaT — Fixed Pipeline (Pico & Nano)
## ✅ Proper Train/Val/Test Split | Best Model Saving | Grad-CAM | Ablation Study

**Fixes applied:**
1. Dataset = 10,000 balanced training images (3333/class)
2. 3-way split: Train (70%) / Val (15%) / Test (15%) — split BEFORE augmentation
3. Augmentation applied ONLY to training split (no leakage)
4. Model selection via validation set; test set used ONLY for final evaluation
5. Best model saved to disk via `torch.save()`
6. Grad-CAM on 20 test images using the saved best model
7. Ablation study on Nano architecture


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1: Installs & Imports
# ══════════════════════════════════════════════════════════════


import os, glob, random, math, warnings, time, copy
import numpy as np
import cv2
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision.transforms import functional as TF

warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = './saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
print(f"Model checkpoints will be saved to: {SAVE_DIR}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2: Dataset Loading & CLAHE Preprocessing
# ══════════════════════════════════════════════════════════════

DATA_DIR = '/kaggle/input/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset'
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 3
CLASS_NAMES = ['Bengin cases', 'Malignant cases', 'Normal cases']

def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def load_and_preprocess(data_dir, class_names, img_size=224):
    images, labels = [], []
    for idx, cls_name in enumerate(class_names):
        cls_path = os.path.join(data_dir, cls_name)
        if not os.path.isdir(cls_path):
            print(f"⚠️ Not found: {cls_path}")
            continue
        file_list = sorted(os.listdir(cls_path))
        print(f"📁 {cls_name}: {len(file_list)} images")
        for fname in file_list:
            fpath = os.path.join(cls_path, fname)
            try:
                img = cv2.imread(fpath)
                if img is None: continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
                img = apply_clahe(img)
                img = img.astype(np.float32) / 255.0
                images.append(img)
                labels.append(idx)
            except Exception as e:
                continue
    return np.array(images), np.array(labels)

all_images, all_labels = load_and_preprocess(DATA_DIR, CLASS_NAMES, IMG_SIZE)
print(f"\n✅ Total: {all_images.shape[0]} | Shape: {all_images.shape[1:]}")
print(f"📊 Distribution: {dict(Counter(all_labels))}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3: PROPER 3-Way Split — BEFORE Any Augmentation
# FIX: Split original data into train(70%) / val(15%) / test(15%)
#      Augmentation is applied ONLY to train split afterward.
# ══════════════════════════════════════════════════════════════

# Step 1: Split off test set (15%) from original data
X_trainval, X_test, y_trainval, y_test = train_test_split(
    all_images, all_labels,
    test_size=0.15,
    random_state=SEED,
    stratify=all_labels
)

# Step 2: Split remaining into train (~70% overall) and val (~15% overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.176,   # 0.176 of 85% ≈ 15% of total
    random_state=SEED,
    stratify=y_trainval
)

print(f"📌 Train: {X_train.shape[0]} images  (original, before augmentation)")
print(f"📌 Val:   {X_val.shape[0]} images   (original, NO augmentation)")
print(f"📌 Test:  {X_test.shape[0]} images  (original, NO augmentation)")
print(f"\n📊 Train dist: {dict(Counter(y_train))}")
print(f"📊 Val dist:   {dict(Counter(y_val))}")
print(f"📊 Test dist:  {dict(Counter(y_test))}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4: Balanced Augmentation — ONLY on Training Split
# FIX: Val and Test are NEVER augmented or touched.
# Target: ~10,000 training images (3333 per class = 9999)
# ══════════════════════════════════════════════════════════════

def augment_image(image_np):
    img = Image.fromarray((image_np * 255).astype(np.uint8))
    angle = random.uniform(-20, 20)
    img = TF.rotate(img, angle, fill=0)
    max_shift = int(IMG_SIZE * 0.1)
    dx, dy = random.randint(-max_shift, max_shift), random.randint(-max_shift, max_shift)
    img = TF.affine(img, angle=0, translate=(dx, dy), scale=1.0, shear=0)
    shear_deg = random.uniform(-10, 10)
    img = TF.affine(img, angle=0, translate=(0, 0), scale=1.0, shear=shear_deg)
    scale = random.uniform(0.9, 1.1)
    new_size = int(IMG_SIZE * scale)
    img = img.resize((new_size, new_size), Image.BILINEAR)
    img = TF.center_crop(img, [IMG_SIZE, IMG_SIZE])
    if random.random() > 0.5:
        img = TF.hflip(img)
    return np.array(img).astype(np.float32) / 255.0

TARGET_TOTAL = 10000
TARGET_PER_CLASS = TARGET_TOTAL // NUM_CLASSES  # 3333

aug_images, aug_labels = [], []
train_counts = Counter(y_train)
print(f"📊 Original train distribution: {dict(train_counts)}")
print(f"🎯 Target per class: {TARGET_PER_CLASS}\n")

for cls_idx in range(NUM_CLASSES):
    cls_mask = y_train == cls_idx
    cls_images = X_train[cls_mask]
    n_current = len(cls_images)
    n_needed = TARGET_PER_CLASS - n_current

    aug_images.extend(cls_images)
    aug_labels.extend([cls_idx] * n_current)

    print(f"  Class {cls_idx} ({CLASS_NAMES[cls_idx]}): {n_current} orig → +{n_needed} aug → {TARGET_PER_CLASS} total")
    for i in range(n_needed):
        src_img = cls_images[i % n_current]
        aug_images.append(augment_image(src_img))
        aug_labels.append(cls_idx)

X_train_aug = np.array(aug_images)
y_train_aug = np.array(aug_labels)

print(f"\n✅ Augmented training set: {len(X_train_aug)} images")
print(f"📊 Balanced distribution: {dict(Counter(y_train_aug))}")
print(f"\n⚠️  Val ({len(X_val)}) and Test ({len(X_test)}) are UNTOUCHED originals — no leakage.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5: Datasets & DataLoaders
# FIX: Each split has its OWN dataset. No random_split on augmented data.
# ══════════════════════════════════════════════════════════════

class LungCancerDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(10),
        ]) if augment else None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32).permute(2, 0, 1)
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

train_dataset = LungCancerDataset(X_train_aug, y_train_aug, augment=True)
val_dataset   = LungCancerDataset(X_val, y_val, augment=False)
test_dataset  = LungCancerDataset(X_test, y_test, augment=False)

NUM_WORKERS = 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"✅ Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"✅ Test:  {len(test_dataset)} samples, {len(test_loader)} batches")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6: Architecture — DSConv, Stem, ResBlock, Downsample
# ══════════════════════════════════════════════════════════════

class DSConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

class DSConvStem(nn.Module):
    def __init__(self, in_ch=3, mid_ch=32, out_ch=64):
        super().__init__()
        self.stem = nn.Sequential(
            DSConv2d(in_ch, mid_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(mid_ch), nn.GELU(),
            DSConv2d(mid_ch, out_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x):
        return self.stem(x)

class DSConvResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            DSConv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels), nn.GELU(),
            DSConv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x):
        return self.block(x) + x

class DSConvDownsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Sequential(
            DSConv2d(in_ch, out_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
        )
    def forward(self, x):
        return self.down(x)

print("✅ Building blocks defined")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7: Mamba Mixer, Self-Attention, MLP Blocks
# ══════════════════════════════════════════════════════════════

class SelectiveScan(nn.Module):
    def __init__(self, d_inner, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.dt_rank = max(1, d_inner // 16)
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):
        B_batch, D, L = x.shape
        x_seq = x.transpose(1, 2)
        x_dbl = self.x_proj(x_seq)
        dt_raw, B_param, C_param = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))
        A = -torch.exp(self.A_log)
        dA = torch.exp(dt.unsqueeze(-1) * A)
        dBx = dt.unsqueeze(-1) * B_param.unsqueeze(2)
        h = torch.zeros(B_batch, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dBx[:, t] * x_seq[:, t, :, None]
            yt = (h * C_param[:, t, None, :]).sum(-1)
            ys.append(yt)
        y = torch.stack(ys, dim=2)
        return y + self.D.unsqueeze(0).unsqueeze(-1) * x

class MambaVisionMixer(nn.Module):
    def __init__(self, dim, d_state=16, kernel_size=3):
        super().__init__()
        half = dim // 2
        self.proj_x = nn.Linear(dim, half, bias=False)
        self.conv1d_x = nn.Conv1d(half, half, kernel_size, padding='same', groups=half)
        self.ssm = SelectiveScan(half, d_state)
        self.proj_z = nn.Linear(dim, half, bias=False)
        self.conv1d_z = nn.Conv1d(half, half, kernel_size, padding='same', groups=half)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        x1 = self.proj_x(x).transpose(1, 2)
        x1 = F.silu(self.conv1d_x(x1))
        x1 = self.ssm(x1)
        x2 = self.proj_z(x).transpose(1, 2)
        x2 = F.silu(self.conv1d_z(x2))
        out = torch.cat([x1, x2], dim=1).transpose(1, 2)
        return self.out_proj(out) + residual

class SelfAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
    def forward(self, x):
        normed = self.norm(x)
        out, _ = self.attn(normed, normed, normed)
        return out + x

class MLPBlock(nn.Module):
    def __init__(self, dim, expand=4, dropout=0.1):
        super().__init__()
        hidden = dim * expand
        self.net = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x) + x

class MambaVisionLayer(nn.Module):
    def __init__(self, dim, mixer_type='mamba', num_heads=4, d_state=16, mlp_expand=4, dropout=0.1):
        super().__init__()
        if mixer_type == 'mamba':
            self.mixer = MambaVisionMixer(dim, d_state=d_state)
        else:
            self.mixer = SelfAttentionBlock(dim, num_heads=num_heads, dropout=dropout)
        self.mlp = MLPBlock(dim, expand=mlp_expand, dropout=dropout)
    def forward(self, x):
        x = self.mixer(x)
        x = self.mlp(x)
        return x

print("✅ Mamba/Attention/MLP blocks defined")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 8: Complete DSMaT Architecture
# ══════════════════════════════════════════════════════════════

class DSMaT(nn.Module):
    def __init__(self, stem_channels=(32, 64), stage_channels=(96, 192, 256, 384),
                 stage_depths=(2, 2, 4, 4), sa_heads=(4, 8), d_state=16,
                 mlp_expand=4, num_classes=3, dropout=0.2):
        super().__init__()
        stem_mid, stem_out = stem_channels

        self.stem = DSConvStem(3, stem_mid, stem_out)

        self.down1 = DSConvDownsample(stem_out, stage_channels[0])
        self.stage1 = nn.Sequential(*[DSConvResBlock(stage_channels[0]) for _ in range(stage_depths[0])])

        self.down2 = DSConvDownsample(stage_channels[0], stage_channels[1])
        self.stage2 = nn.Sequential(*[DSConvResBlock(stage_channels[1]) for _ in range(stage_depths[1])])

        self.down3 = DSConvDownsample(stage_channels[1], stage_channels[2])
        n3 = stage_depths[2]; n3m = n3 // 2; n3a = n3 - n3m
        s3 = []
        for _ in range(n3m):
            s3.append(MambaVisionLayer(stage_channels[2], 'mamba', d_state=d_state,
                                       mlp_expand=mlp_expand, dropout=dropout))
        for _ in range(n3a):
            s3.append(MambaVisionLayer(stage_channels[2], 'attention', num_heads=sa_heads[0],
                                       mlp_expand=mlp_expand, dropout=dropout))
        self.stage3 = nn.ModuleList(s3)

        self.down4 = DSConvDownsample(stage_channels[2], stage_channels[3])
        n4 = stage_depths[3]; n4m = n4 // 2; n4a = n4 - n4m
        s4 = []
        for _ in range(n4m):
            s4.append(MambaVisionLayer(stage_channels[3], 'mamba', d_state=d_state,
                                       mlp_expand=mlp_expand, dropout=dropout))
        for _ in range(n4a):
            s4.append(MambaVisionLayer(stage_channels[3], 'attention', num_heads=sa_heads[1],
                                       mlp_expand=mlp_expand, dropout=dropout))
        self.stage4 = nn.ModuleList(s4)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.LayerNorm(stage_channels[3]),
            nn.Dropout(dropout),
            nn.Linear(stage_channels[3], num_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.stem(x)
        x = self.down1(x); x = self.stage1(x)
        x = self.down2(x); x = self.stage2(x)
        x = self.down3(x)
        B3, C3, H3, W3 = x.shape
        x = x.flatten(2).transpose(1, 2)
        for layer in self.stage3:
            x = layer(x)
        x = x.transpose(1, 2).reshape(B, C3, H3, W3)
        x = self.down4(x)
        B4, C4, H4, W4 = x.shape
        x = x.flatten(2).transpose(1, 2)
        for layer in self.stage4:
            x = layer(x)
        x = x.transpose(1, 2).reshape(B, C4, H4, W4)
        x = self.pool(x).flatten(1)
        return self.head(x)

print("✅ DSMaT model defined")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 9: Model Configurations — Pico & Nano Only
# ══════════════════════════════════════════════════════════════

MODEL_CONFIGS = {
    'DSMaT-Pico': dict(
        stem_channels=(16, 32), stage_channels=(48, 96, 128, 192),
        stage_depths=(1, 1, 2, 2), sa_heads=(4, 4), mlp_expand=2, dropout=0.2,
    ),
    'DSMaT-Nano': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
    ),
}

def build_model(name, num_classes=3):
    cfg = {k: v for k, v in MODEL_CONFIGS[name].items()}
    return DSMaT(num_classes=num_classes, **cfg).to(DEVICE)

for name in MODEL_CONFIGS:
    m = build_model(name)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:20s} | Params: {n_params:>10,} ({n_params/1e6:.2f}M)")
    del m


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 10: Training & Evaluation Utilities — WITH BEST MODEL SAVING
# FIX: Train on train_loader, validate on val_loader.
#      Test set is ONLY used for final evaluation after training.
#      Best model checkpoint is saved to disk via torch.save().
# ══════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sum += loss.item() * labels.size(0)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (loss_sum / total, correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

def train_model(model, model_name, train_loader, val_loader, epochs=50, lr=1e-3):
    """Train with validation-based model selection. Saves best checkpoint to disk."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    save_path = os.path.join(SAVE_DIR, f'{model_name}_best.pth')

    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # ── SAVE BEST MODEL ──
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'model_name': model_name,
            }, save_path)
            marker = ' ★ saved'
        else:
            marker = ''

        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"  Ep {epoch+1:3d}/{epochs} | Tr {tr_loss:.4f}/{tr_acc:.4f} "
                  f"| Val {val_loss:.4f}/{val_acc:.4f} | {time.time()-t0:.1f}s{marker}")

    # ── RELOAD BEST MODEL from disk ──
    checkpoint = torch.load(save_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\n  💾 Best model loaded from: {save_path}")
    print(f"     Best epoch: {checkpoint['epoch']} | Val Acc: {checkpoint['val_acc']*100:.2f}%")

    return history, best_val_acc, save_path

print("✅ Training utilities defined (with torch.save best model)")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 11: Train DSMaT-Pico and DSMaT-Nano
# ══════════════════════════════════════════════════════════════

EPOCHS = 50
LR = 1e-3
results = {}

for name in MODEL_CONFIGS.keys():
    print("=" * 70)
    print(f"  TRAINING: {name}")
    print("=" * 70)

    model = build_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,} ({n_params/1e6:.2f}M)")

    history, best_val_acc, save_path = train_model(
        model, name, train_loader, val_loader, epochs=EPOCHS, lr=LR
    )

    # Final evaluation on TEST set (only after training is complete)
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
        model, test_loader, criterion, DEVICE
    )

    results[name] = {
        'history': history,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'test_preds': test_preds,
        'test_labels': test_labels,
        'test_probs': test_probs,
        'params': n_params,
        'save_path': save_path,
    }
    print(f"\n  ✅ Best Val Acc: {best_val_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")
    print(f"  💾 Saved to: {save_path}\n")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Identify overall best model ──
best_model_name = max(results, key=lambda n: results[n]['test_acc'])
print("=" * 70)
print(f"🏆 BEST MODEL: {best_model_name} | Test Acc: {results[best_model_name]['test_acc']*100:.2f}%")
print(f"   Checkpoint: {results[best_model_name]['save_path']}")
print("=" * 70)


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 12: Accuracy & Loss Curves
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, name in enumerate(MODEL_CONFIGS.keys()):
    h = results[name]['history']
    epochs_range = range(1, len(h['train_loss']) + 1)

    axes[0, i].plot(epochs_range, h['train_loss'], label='Train Loss')
    axes[0, i].plot(epochs_range, h['val_loss'], label='Val Loss')
    axes[0, i].set_title(f'{name} — Loss', fontweight='bold')
    axes[0, i].set_xlabel('Epoch'); axes[0, i].set_ylabel('Loss')
    axes[0, i].legend(); axes[0, i].grid(True, alpha=0.3)

    axes[1, i].plot(epochs_range, h['train_acc'], label='Train Acc')
    axes[1, i].plot(epochs_range, h['val_acc'], label='Val Acc')
    axes[1, i].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[1, i].set_xlabel('Epoch'); axes[1, i].set_ylabel('Accuracy')
    axes[1, i].legend(); axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 13: Test-Set Evaluation — Confusion Matrix, Report, ROC
# ══════════════════════════════════════════════════════════════

class_labels = ['Benign', 'Malignant', 'Normal']

for name in MODEL_CONFIGS.keys():
    r = results[name]
    preds, labels, probs = r['test_preds'], r['test_labels'], r['test_probs']

    print("=" * 70)
    print(f"  TEST EVALUATION: {name} | Test Acc: {r['test_acc']*100:.2f}%")
    print(f"  Loaded from: {r['save_path']}")
    print("=" * 70)
    print(classification_report(labels, preds, target_names=class_labels))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
    axes[0].set_title(f'{name} — Confusion Matrix')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    # ROC curves
    y_bin = label_binarize(labels, classes=[0, 1, 2])
    for cls_i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, cls_i], probs[:, cls_i])
        roc_auc = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, label=f'{class_labels[cls_i]} (AUC={roc_auc:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[1].set_title(f'{name} — ROC Curves')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'test_eval_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 14: Grad-CAM — 20 Test Images Using SAVED BEST MODEL
# Loads the best checkpoint from disk for each model.
# ══════════════════════════════════════════════════════════════

class GradCAM:
    """Grad-CAM targeting the last conv stage (stage2 output)."""
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        self.model.stage2.register_forward_hook(self._fwd_hook)
        self.model.stage2.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, input, output):
        self.activations = output.detach()

    def _bwd_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot)

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class

# ── Generate Grad-CAM for BOTH models using saved checkpoints ──
for name in MODEL_CONFIGS.keys():
    # ── LOAD BEST MODEL FROM DISK ──
    save_path = results[name]['save_path']
    print(f"\n📂 Loading best model: {save_path}")

    model = build_model(name)
    checkpoint = torch.load(save_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(DEVICE).eval()
    print(f"   ✅ Loaded {name} (epoch {checkpoint['epoch']}, val_acc={checkpoint['val_acc']*100:.2f}%)")

    grad_cam = GradCAM(model)

    # Select 20 random test images (deterministic)
    np.random.seed(SEED)
    indices = np.random.choice(len(X_test), 20, replace=False)

    fig, axes = plt.subplots(4, 5, figsize=(25, 20))
    fig.suptitle(f'{name} — Grad-CAM on 20 Test Images (Best Saved Model)', fontsize=18, fontweight='bold')

    for ax_idx, img_idx in enumerate(indices):
        row, col = ax_idx // 5, ax_idx % 5
        img = X_test[img_idx]
        true_label = y_test[img_idx]

        input_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        cam, pred_class = grad_cam.generate(input_tensor)

        ax = axes[row, col]
        ax.imshow(img)
        ax.imshow(cam, cmap='jet', alpha=0.4)
        color = 'green' if pred_class == true_label else 'red'
        ax.set_title(f"True: {class_labels[true_label]}\nPred: {class_labels[pred_class]}",
                     fontsize=10, color=color, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'gradcam_{name}_best.png', dpi=150, bbox_inches='tight')
    plt.show()

    del model, grad_cam
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 15: Ablation Study — Using DSMaT-Nano Architecture
# Variants: Full | No Mamba | No Attention | Conv-Only
# Each ablation variant's best model is also saved.
# ══════════════════════════════════════════════════════════════

ABLATION_CONFIGS = {
    'Full DSMaT-Nano': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
        ablation=None,
    ),
    'No Mamba (Attn Only)': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
        ablation='attn_only',
    ),
    'No Attention (Mamba Only)': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
        ablation='mamba_only',
    ),
    'Conv-Only (No Mamba/Attn)': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
        ablation='conv_only',
    ),
}

def build_ablation_model(name, num_classes=3):
    cfg_full = ABLATION_CONFIGS[name]
    ablation = cfg_full.get('ablation', None)
    cfg = {k: v for k, v in cfg_full.items() if k != 'ablation'}
    model = DSMaT(num_classes=num_classes, **cfg).to(DEVICE)

    if ablation == 'conv_only':
        for layer in model.stage3:
            layer.mixer = nn.Identity()
        for layer in model.stage4:
            layer.mixer = nn.Identity()
    elif ablation == 'attn_only':
        n3 = len(model.stage3)
        for i in range(n3 // 2):
            model.stage3[i].mixer = nn.Identity()
        n4 = len(model.stage4)
        for i in range(n4 // 2):
            model.stage4[i].mixer = nn.Identity()
    elif ablation == 'mamba_only':
        n3 = len(model.stage3)
        for i in range(n3 // 2, n3):
            model.stage3[i].mixer = nn.Identity()
        n4 = len(model.stage4)
        for i in range(n4 // 2, n4):
            model.stage4[i].mixer = nn.Identity()
    return model

# ── Train all ablation variants ──
ABLATION_EPOCHS = 30
ablation_results = {}

for name in ABLATION_CONFIGS.keys():
    print("=" * 70)
    print(f"  ABLATION: {name}")
    print("=" * 70)

    model = build_ablation_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,} ({n_params/1e6:.2f}M)")

    # Use same train_model function — saves best to disk
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '-')
    history, best_val_acc, save_path = train_model(
        model, f'ablation_{safe_name}', train_loader, val_loader,
        epochs=ABLATION_EPOCHS, lr=LR
    )

    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, _, _, _ = evaluate(model, test_loader, criterion, DEVICE)

    ablation_results[name] = {
        'params': n_params, 'best_val_acc': best_val_acc,
        'test_acc': test_acc, 'history': history, 'save_path': save_path,
    }
    print(f"\n  ✅ Val Acc: {best_val_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")
    print(f"  💾 Saved to: {save_path}\n")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Ablation Summary Table ──
print("\n" + "=" * 90)
print(f"{'Ablation Variant':35s} {'Params':>10s} {'Val Acc':>10s} {'Test Acc':>10s} {'Checkpoint':>20s}")
print("-" * 90)
for name, r in ablation_results.items():
    ckpt = os.path.basename(r['save_path'])
    print(f"{name:35s} {r['params']:>10,} {r['best_val_acc']*100:>9.2f}% {r['test_acc']*100:>9.2f}% {ckpt:>20s}")
print("=" * 90)


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 16: Ablation Study Visualization
# ══════════════════════════════════════════════════════════════

names = list(ablation_results.keys())
test_accs = [ablation_results[n]['test_acc'] * 100 for n in names]
val_accs = [ablation_results[n]['best_val_acc'] * 100 for n in names]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, val_accs, width, label='Val Acc', color='#3498db')
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Acc', color='#e74c3c')

ax.set_xlabel('Ablation Variant')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation Study — Component Contribution (Nano Architecture)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 17: List All Saved Model Checkpoints
# ══════════════════════════════════════════════════════════════

print("\n📦 All saved model checkpoints:")
print("=" * 70)
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    ckpt = torch.load(fpath, map_location='cpu')
    val_acc = ckpt.get('val_acc', 0) * 100
    epoch = ckpt.get('epoch', '?')
    mname = ckpt.get('model_name', f)
    print(f"  {f:50s} {size_mb:6.1f} MB | Epoch {epoch:>3} | Val Acc {val_acc:.2f}%")

print("\n🏆 Best overall model checkpoint:")
print(f"   {results[best_model_name]['save_path']}")
print(f"   Model: {best_model_name} | Test Acc: {results[best_model_name]['test_acc']*100:.2f}%")

print("\n💡 To load a saved model later:")
print("""
    model = build_model('DSMaT-Nano')
    checkpoint = torch.load('./saved_models/DSMaT-Nano_best.pth', map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
""")
